# HuggingFace Pipeline Use

1. 주제를 선정한다.
2. 사전 학습(또는 전이 학습된) 모델을 HuggingFace로부터 Pipeline을 이용해 로드해 온다.
3. 내가 원하는 주제에 부합하는 시스템이 동작하도록 한다.
- console 에서 진행 OK
- streamlit 구현 OK

In [ ]:
from transformers import pipeline
import numpy as np

# 사과문 분류 모델
classifier = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",
    framework="pt"
)

# 문장 임베딩 모델
embedder = pipeline(
    "feature-extraction",
    model="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    framework="pt"
)

text = """
저희 삼성서울병원이 메르스 감염과 확산을 막지 못해 국민 여러분께 너무 큰 고통과 걱정을 끼쳐 드렸습니다. 머리 숙여 사죄합니다. (사죄의 인사)

특히 메르스로 인해 유명을 달리하신 분들과 유족분들, 아직 치료 중이신 환자분들, 예기치 않은 격리조치로 불편을 겪으신 분들께 죄송합니다.

저의 아버님께서도 1년 넘게 병원에 누워 계십니다. 환자 분들과 가족분들께서 겪으신 불안과 고통을 조금이나마 이해하고 있습니다. 환자분들은 저희가 끝까지 책임지고 치료해 드리겠습니다.

관계 당국과도 긴밀히 협조해 메르스 사태가 이른 시일 안에 완전히 해결되도록 모든 힘을 다 하겠습니다.

저희는 국민 여러분의 기대와 신뢰에 미치지 못했습니다. 제 자신 참담한 심정입니다. 책임을 통감합니다. 사태가 수습되는 대로 병원을 대대적으로 혁신하겠습니다.

어떻게 이런 일이 발생했는지 철저히 조사하고 재발방지를 위해 최선의 노력을 다하겠습니다.

이번 일을 계기로 응급실을 포함한 진료환경을 개선하고 부족했던 음압 병실도 충분히 갖춰서 환자 분들께서 안심하고 치료받을 수 있는 환경을 만들겠습니다.

저희는 앞으로 이런 감염 질환에 대처하기 위해 예방 활동과 함께 백신과 치료제 개발을 적극 지원하겠습니다.

(또한) 말씀 드리기 송구스럽지만 의료진은 벌써 한 달 이상 밤낮 없이 치료와 간호에 헌신하고 있습니다. 이분들에게 격려와 성원을 부탁드립니다.

메르스로 큰 고통을 겪고 계신 환자분들의 조속한 쾌유를 기원하면서 다시 한번 진심으로 사과 드립니다.
"""

# 분류에 사용할 라벨
candidate_labels = [
    "자신의 잘못을 명확히 인정하고 변명 없이 사과하는 글",
    "잘못 인정보다 오해 해소와 상황 설명에 더 집중하는 글"
]

hypothesis_template = "이 글은 {}에 가깝습니다."

# 최종 결과값 매핑
result_map = {
    "자신의 잘못을 명확히 인정하고 변명 없이 사과하는 글": "진실된 사과문",
    "잘못 인정보다 오해 해소와 상황 설명에 더 집중하는 글": "4과문"
}

# 최종 결과별로 보여줄 요약 후보 문장
candidates = {
    "진실된 사과문": [
        "잘못을 직접 인정하고 책임을 수용하는 사과문",
        "사과와 재발 방지 의지가 비교적 분명한 글",
        "문제의 원인을 인정하고 사과의 뜻을 밝힌 글",
        "책임 회피보다 잘못 인정이 중심인 사과문"
    ],
    "4과문": [
        "해명과 변명 비중이 큰 사과문",
        "직접적 책임 인정보다 형식적 표현이 많은 글",
        "사과보다 오해 해소와 해명에 무게가 실린 글",
        "책임 인정 표현이 약하고 해명 위주로 작성된 글"
    ]
}

# 여러 토큰 벡터의 평균을 내서 문장 벡터 1개로 만드는 함수
def to_vec(x):
    return np.array(x).mean(axis=0)

# 두 벡터가 얼마나 비슷한지 계산하는 함수
def sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 사과문 분류
result = classifier(
    text,
    candidate_labels=candidate_labels,
    hypothesis_template=hypothesis_template
)

label = result_map[result["labels"][0]]

# 입력 사과문을 벡터로 변환
text_vec = to_vec(embedder(text)[0])

best_summary = ""
best_score = -1

# 분류 결과에 맞는 후보 문장들 중 가장 비슷한 문장 선택
for s in candidates[label]:
    score = sim(text_vec, to_vec(embedder(s)[0]))
    if score > best_score:
        best_score = score
        best_summary = s

print(f"결과값 : {label}")
print(f"요약 : {best_summary}")

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Device set to use cpu


결과값 : 진실된 사과문
요약 : 문제의 원인을 인정하고 사과의 뜻을 밝힌 글


예시 (르노 삼성자동차 논란 사과문)


저는 특정 손 모양이 문제가 되는 혐오의 행동이라는 것을 알고 있었지만, 정작 제가 제작한 영상에서 표현한 손 모양이 그러한 의미로 해석될 수 있다는 것을 미처 인식하지 못했습니다.

이는 저의 부주의였으며, 다시 한번 죄송하다는 말씀드립니다. 앞으로 더 주의 깊게 행동하겠습니다.

저는 일반인이고 그저 직장인입니다. 직접 제 얼굴이 그대로 노출되는 영상 콘텐츠의 특성상 문제가 될 수 있는 어떤 행동을 의도를 가지고 한다는 것은 저 스스로도 상상하기 어렵습니다. 이러한 사정을 부디 너그럽게 이해해 주시기를 부탁드립니다.

힘든 시간을 보내고 있습니다. 인신공격을 멈추어 주시기를 간곡히 부탁드립니다.

또한 고객님들께 더 나은 가치를 제공하기 위해 밤낮없이 헌신해온 회사와 임직원 동료분들 또한 제 실수로 인해 힘든 시간을 보내고 있습니다. 송구스러운 마음뿐입니다. 죄송합니다.

앞으로 이와 같은 실수가 재발하지 않도록 더욱 신중하게 행동하겠습니다.

In [25]:
from transformers import pipeline
import numpy as np

classifier = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",
    framework="pt"
)

embedder = pipeline(
    "feature-extraction",
    model="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    framework="pt"
)

candidate_labels = [
    "자신의 잘못을 명확히 인정하고 변명 없이 사과하는 글",
    "잘못 인정보다 오해 해소와 상황 설명에 더 집중하는 글"
]
hypothesis_template = "이 글은 {}에 가깝습니다."

result_map = {
    "자신의 잘못을 명확히 인정하고 변명 없이 사과하는 글": "진실된 사과문",
    "잘못 인정보다 오해 해소와 상황 설명에 더 집중하는 글": "4과문"
}

candidates = {
    "진실된 사과문": [
        "잘못을 직접 인정하고 책임을 분명히 밝힌 글",
        "사과와 반성이 중심이 되는 사과문",
        "구체적인 책임 인정이 드러나는 글"
    ],
    "4과문": [
        "오해와 상황 설명 비중이 큰 사과문",
        "직접적 책임 인정보다 해명 표현이 많은 글",
        "잘못 설명보다 관계와 상황 해명이 앞서는 글",
        "사과보다는 오해 해소에 무게가 실린 글"
    ]
}

def to_vec(x):
    return np.array(x).mean(axis=0)

def sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

while True:
    text = input("\n사과문을 입력하세요(종료를 입력하여 끝내기): ").strip()

    if text == "종료":
        break
    if not text:
        continue

    result = classifier(
        text,
        candidate_labels=candidate_labels,
        hypothesis_template=hypothesis_template
    )

    label = result_map[result["labels"][0]]
    text_vec = to_vec(embedder(text)[0])

    best_summary = ""
    best_score = -1

    for s in candidates[label]:
        score = sim(text_vec, to_vec(embedder(s)[0]))
        if score > best_score:
            best_score = score
            best_summary = s

    print(f"\n결과값 : {label}")
    print(f"요약 : {best_summary}")

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Device set to use cpu



결과값 : 4과문
요약 : 직접적 책임 인정보다 해명 표현이 많은 글
